In [137]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import shutil
from pathlib import Path
import tensorflow as tf
import numpy as np
import pathlib
import matplotlib.pyplot as plt
import mlflow
import dagshub
from dotenv import load_dotenv
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers.experimental.preprocessing import RandomZoom
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

### Counting the number of images present for each label

In [138]:
parent_folder = "Datasets/6_Garbage_Class"
image_extensions = ('.jpg', '.jpeg', '.png')

data_dir = pathlib.Path(parent_folder)

for folder_name in os.listdir(parent_folder):
    img_count = len(list(data_dir.glob(f'{folder_name}/*')))
    print(f'{folder_name}: {img_count} images')

cardboard: 403 images
glass: 501 images
metal: 410 images
paper: 594 images
plastic: 482 images
trash: 137 images


# Train, Validation and Test Splitting

In [139]:
"""def split_dataset(parent_folder):

    # percentages
    train_ratio = 0.7
    val_ratio = 0.15
    test_ratio = 0.15

    # get base folder (Datasets/)
    base_folder = os.path.dirname(parent_folder)

    train_path = os.path.join(base_folder, "Train")
    val_path = os.path.join(base_folder, "Val")
    test_path = os.path.join(base_folder, "Test")

    # create main folders
    os.makedirs(train_path, exist_ok=True)
    os.makedirs(val_path, exist_ok=True)
    os.makedirs(test_path, exist_ok=True)

    # loop through each class
    for label in os.listdir(parent_folder):

        label_path = os.path.join(parent_folder, label)

        if not os.path.isdir(label_path):
            continue

        print("Processing:", label)

        images = os.listdir(label_path)

        # remove hidden files if any
        images = [img for img in images if not img.startswith('.')]

        random.shuffle(images)

        total = len(images)

        # calculate split counts
        train_count = int(total * train_ratio)
        val_count = int(total * val_ratio)

        # split images
        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]

        # create label folders
        train_label = os.path.join(train_path, label)
        val_label = os.path.join(val_path, label)
        test_label = os.path.join(test_path, label)

        os.makedirs(train_label, exist_ok=True)
        os.makedirs(val_label, exist_ok=True)
        os.makedirs(test_label, exist_ok=True)

        # copy files
        for img in train_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(train_label, img))

        for img in val_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(val_label, img))

        for img in test_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(test_label, img))

        print(f"{label} → Total:{total} | Train:{len(train_images)}, Val:{len(val_images)}, Test:{len(test_images)}")

    print("\nDone!")"""

'def split_dataset(parent_folder):\n\n    # percentages\n    train_ratio = 0.7\n    val_ratio = 0.15\n    test_ratio = 0.15\n\n    # get base folder (Datasets/)\n    base_folder = os.path.dirname(parent_folder)\n\n    train_path = os.path.join(base_folder, "Train")\n    val_path = os.path.join(base_folder, "Val")\n    test_path = os.path.join(base_folder, "Test")\n\n    # create main folders\n    os.makedirs(train_path, exist_ok=True)\n    os.makedirs(val_path, exist_ok=True)\n    os.makedirs(test_path, exist_ok=True)\n\n    # loop through each class\n    for label in os.listdir(parent_folder):\n\n        label_path = os.path.join(parent_folder, label)\n\n        if not os.path.isdir(label_path):\n            continue\n\n        print("Processing:", label)\n\n        images = os.listdir(label_path)\n\n        # remove hidden files if any\n        images = [img for img in images if not img.startswith(\'.\')]\n\n        random.shuffle(images)\n\n        total = len(images)\n\n        # c

In [140]:
"""parent_folder = "Datasets/6_Garbage_Class"
split_dataset(parent_folder)"""

'parent_folder = "Datasets/6_Garbage_Class"\nsplit_dataset(parent_folder)'

In [141]:
"""train_path = 'Datasets/Train'
data_dir = pathlib.Path(train_path)

for folder_name in os.listdir(train_path):
    img_count = len(list(data_dir.glob(f'{folder_name}/*')))
    print(f'{folder_name}: {img_count} images')

total_image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total Training Images: {total_image_count}")"""

'train_path = \'Datasets/Train\'\ndata_dir = pathlib.Path(train_path)\n\nfor folder_name in os.listdir(train_path):\n    img_count = len(list(data_dir.glob(f\'{folder_name}/*\')))\n    print(f\'{folder_name}: {img_count} images\')\n\ntotal_image_count = len(list(data_dir.glob(\'*/*.jpg\')))\nprint(f"Total Training Images: {total_image_count}")'

In [142]:
"""battery = list(data_dir.glob('battery/*'))
Image.open(str(battery[0]))"""

"battery = list(data_dir.glob('battery/*'))\nImage.open(str(battery[0]))"

In [143]:
"""train_object_image_dict = {}

for folder_name in os.listdir(train_path):
    train_object_image_dict[folder_name] = list(data_dir.glob(f"{folder_name}/*"))
"""

'train_object_image_dict = {}\n\nfor folder_name in os.listdir(train_path):\n    train_object_image_dict[folder_name] = list(data_dir.glob(f"{folder_name}/*"))\n'

In [144]:
"""object_labels_dict = {}
label = 0
for folder_name in os.listdir(train_path):
    object_labels_dict[folder_name] = label
    label += 1

object_labels_dict"""

'object_labels_dict = {}\nlabel = 0\nfor folder_name in os.listdir(train_path):\n    object_labels_dict[folder_name] = label\n    label += 1\n\nobject_labels_dict'

### Resize Images

In [145]:
"""X_train, y_train = [], []

for object_name, images in train_object_image_dict.items():
    for image in images:
        img = cv2.imread(image)
        resized_img = cv2.resize(img, (256, 256))
        X_train.append(resized_img)
        y_train.append(object_labels_dict[object_name])"""

'X_train, y_train = [], []\n\nfor object_name, images in train_object_image_dict.items():\n    for image in images:\n        img = cv2.imread(image)\n        resized_img = cv2.resize(img, (256, 256))\n        X_train.append(resized_img)\n        y_train.append(object_labels_dict[object_name])'

In [146]:
"""X_train = np.array(X_train)
y_train = np.array(y_train)"""

'X_train = np.array(X_train)\ny_train = np.array(y_train)'

### Scaling the images

In [147]:
"""X_train_scaled = X_train / 255
X_test_scaled = X_test / 255"""

'X_train_scaled = X_train / 255\nX_test_scaled = X_test / 255'

# Data Pipeline

In [148]:
train_path = "Datasets/6_Train"
val_path = "Datasets/6_Val"
test_path = "Datasets/6_Test"

IMAGE_SIZE = (180, 180)
BATCH_SIZE = 32
SEED = 123

train_ds = image_dataset_from_directory(
  train_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED,
  label_mode='categorical'
)

val_ds = image_dataset_from_directory(
  val_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED,
  label_mode='categorical'
)

test_ds = image_dataset_from_directory(
    test_path,
    image_size=IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    shuffle=False,
    label_mode = None
)

test_ds_with_labels = image_dataset_from_directory(
    test_path,
    image_size=IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
)

# For model performance evaluation
y_test = np.concatenate([y for x, y in test_ds_with_labels])
y_test_labels = np.argmax(y_test, axis=1)

class_names = train_ds.class_names
test_class_names = test_ds_with_labels.class_names


train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

"""train_ds = train_ds.apply(
  tf.data.experimental.ignore_errors()
)"""


Found 1766 files belonging to 6 classes.
Found 377 files belonging to 6 classes.
Found 384 files belonging to 1 classes.
Found 384 files belonging to 6 classes.


'train_ds = train_ds.apply(\n  tf.data.experimental.ignore_errors()\n)'

# CNN Model Pipeline

In [149]:
num_classes = len(class_names)

cnn_params = {
    "model_name": "Convolutional Neural Network",
    "input_shape": (180, 180, 3),
    "conv1_filters": 16,
    "conv2_filters": 32,
    "conv3_filters": 64,
    "kernel_size": 3,
    "dense_units": 128,
    "optimizer": "adam",
    "loss": 'categorical_crossentropy',
    "epochs": 30,
    "early_stopping_patience": 15
}

cnn_model = Sequential([
    Rescaling(1./255, input_shape=cnn_params['input_shape']),
    Conv2D(cnn_params['conv1_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(cnn_params['conv2_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(cnn_params['conv3_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(cnn_params['dense_units'], activation='relu'),
    Dense(num_classes, activation='softmax')
])

cnn_tags = {
    "framework": "tensorflow",
    "model_type": "Convolutional Neural Network"
}

# ResNet50 Model Pipeline

In [150]:
resnet_params ={
    "model_name":"ResNet50",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

resnet_model = Sequential()

pretrained_model = tf.keras.applications.ResNet50(
    include_top=False,
    input_shape=resnet_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

resnet_model.add(pretrained_model)
resnet_model.add(Flatten())
resnet_model.add(Dense(resnet_params['dense_units'], activation='relu'))
resnet_model.add(Dense(len(class_names), activation='softmax'))

resnet50_tags = {
    "framework":"tensorflow",
    "model_type":'ResNet50'
}

# Model Training

In [151]:
models = [
    (
        "Convolutional Neural Network",
        cnn_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        cnn_params,
        cnn_tags
    ),
    (
        "ResNet50",
        resnet_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        resnet_params,
        resnet50_tags
    )
]

In [152]:
model_reports = []
model_history = []

for model_name, model, train_data, val_data, test_data, y_true, model_param, model_tag in models:

  train_ds = train_data
  val_ds = val_data
  test_ds = test_data
  y_test = y_true

  model.compile(
      optimizer=model_param['optimizer'],
      loss=model_param['loss'],
      metrics=['accuracy']
  )

  print(f"{model_name} Compiled")

  early_stop = EarlyStopping(
    monitor='val_loss',
    patience=model_param['early_stopping_patience'],
    restore_best_weights=True
  )

  print(f"{model_name} training started")

  history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=model_param['epochs'],
    callbacks=[early_stop]
  )

  print(f"{model_name} Trained Successfully!")

  model_history.append(history)

  predictions = model.predict(test_ds)
  y_pred = np.argmax(predictions, axis=1)

  report = report = classification_report(y_test_labels, y_pred, target_names=test_class_names, output_dict=True)
  model_reports.append(report)

  print("Reports and history added!")

Convolutional Neural Network Compiled
Convolutional Neural Network training started
Epoch 1/30
56/56 [==============================] - 2s 28ms/step - loss: 1.5785 - accuracy: 0.3601 - val_loss: 1.4304 - val_accuracy: 0.4403
Epoch 2/30
56/56 [==============================] - 1s 25ms/step - loss: 1.2573 - accuracy: 0.5045 - val_loss: 1.2415 - val_accuracy: 0.4987
Epoch 3/30
56/56 [==============================] - 1s 25ms/step - loss: 1.0368 - accuracy: 0.5985 - val_loss: 1.2400 - val_accuracy: 0.5411
Epoch 4/30
56/56 [==============================] - 1s 25ms/step - loss: 0.8766 - accuracy: 0.6891 - val_loss: 1.1981 - val_accuracy: 0.6021
Epoch 5/30
56/56 [==============================] - 1s 25ms/step - loss: 0.6879 - accuracy: 0.7542 - val_loss: 1.1743 - val_accuracy: 0.6446
Epoch 6/30
56/56 [==============================] - 1s 25ms/step - loss: 0.5198 - accuracy: 0.8177 - val_loss: 1.1700 - val_accuracy: 0.6366
Epoch 7/30
56/56 [==============================] - 1s 24ms/step - los

# Model Tracking in Dagshub

In [153]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Garbage-Image-Classifier', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
  model_name = element[0]
  model = element[1]
  train_ds = element[2]
  val_ds = element[3]
  test_ds = element[4]
  y_test = element[5]
  param = element[6]
  tag = element[7]

  model_report = model_reports[i]
  history = model_history[i]


  with mlflow.start_run(run_name=model_name):

    #--------------------------------------
    # Data Pipeline logging
    #--------------------------------------

    mlflow.set_tags(tag)

    mlflow.log_param("train_data_path", train_path)
    mlflow.log_param("val_data_path", val_path)
    mlflow.log_param("Seed", SEED)
    mlflow.log_param("image_height", IMAGE_SIZE[0])
    mlflow.log_param("image_width", IMAGE_SIZE[1])
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("class_names", class_names)
    mlflow.log_params(param)

    print(f"{model_name} parameters logged!")

    #-----------------------------------------------------
    # Model training and validation performance tracking
    #-----------------------------------------------------

    for epoch in range(len(history.history['loss'])):
      mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
      mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)
      mlflow.log_metric("val_loss", history.history['val_loss'][epoch], step=epoch)
      mlflow.log_metric("val_accuracy", history.history['val_accuracy'][epoch], step=epoch)

    #------------------------------------------------------
    # Log Model
    #------------------------------------------------------

    model_info = mlflow.tensorflow.log_model(
      model, 
      artifact_path=model_name
    )

    print(f"{model_name} model logged!")

    #------------------------------------------------------
    # Model testing performance tracking
    #------------------------------------------------------

    # ✅ Log best epoch metrics explicitly (for clean experiment table comparison)
    best_epoch = np.argmin(history.history['val_loss'])
    mlflow.log_metric("best_epoch", best_epoch)
    mlflow.log_metric("best_val_loss", history.history['val_loss'][best_epoch])
    mlflow.log_metric("best_val_accuracy", history.history['val_accuracy'][best_epoch])
    mlflow.log_metric("best_train_accuracy", history.history['accuracy'][best_epoch])

    # ✅ Link test metrics to the LoggedModel using model_id
    mlflow_metrics = {}
    for label, metrics in model_report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                if metric_name != "support":
                    mlflow_metrics[f"{label}_{metric_name}"] = float(value)
        else:
            mlflow_metrics[label] = float(metrics)

    # Pass model_id to link metrics to the model in the Models tab
    mlflow.log_metrics(mlflow_metrics, model_id=model_info.model_id)

    print(f"{model_name} metrics logged")


Initialized MLflow to track repo "JS-Tharun/Garbage-Image-Classifier"

Repository JS-Tharun/Garbage-Image-Classifier initialized!

Convolutional Neural Network parameters logged!


2026/04/19 15:40:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:40:12 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpdw18df_v\model\data\model\assets


...

Convolutional Neural Network model logged!
Convolutional Neural Network metrics logged
🏃 View run Convolutional Neural Network at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1/runs/2dd45fa0a01649f3b38a47f14e39414c
🧪 View experiment at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1
ResNet50 parameters logged!


2026/04/19 15:41:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:41:31 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpgr5rdvwn\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpgr5rdvwn\model\data\model\assets


ResNet50 model logged!
ResNet50 metrics logged
🏃 View run ResNet50 at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1/runs/b1d72ffc4898426483ae13cded9410d9
🧪 View experiment at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1
